# Intention Collapse: Consolidate Results v3.0

This notebook loads all experiment results and generates paper-ready outputs.

**Run AFTER completing all 9 experiments.**

---
## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/intention_collapse_v3'
OUTPUT_PATH = os.path.join(DRIVE_PATH, 'paper_outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"✓ Results path: {DRIVE_PATH}")
print(f"✓ Output path: {OUTPUT_PATH}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

print("✓ Imports complete")

---
## 2. Load All Results

In [ ]:
# Define experiment matrix
MODELS = ['mistral', 'llama', 'qwen']
BENCHMARKS = ['gsm8k', 'math', 'arc']
CONDITIONS = ['baseline', 'enhanced', 'babble']

# Model display names
MODEL_NAMES = {
    'mistral': 'Mistral-7B',
    'llama': 'Llama-3.1-8B',
    'qwen': 'Qwen-2.5-7B'
}

# Load all results
all_results = {}
missing = []

for model in MODELS:
    for benchmark in BENCHMARKS:
        key = (model, benchmark)
        filepath = os.path.join(DRIVE_PATH, f"{model}_{benchmark}_results.json")
        
        if os.path.exists(filepath):
            with open(filepath, 'r') as f:
                all_results[key] = json.load(f)
            print(f"✓ Loaded {model}/{benchmark}")
        else:
            missing.append(key)
            print(f"⚠ Missing {model}/{benchmark}")

print(f"\nLoaded: {len(all_results)}/9 experiments")
if missing:
    print(f"Missing: {missing}")

In [ ]:
# Verify consistent dataset indices
print("\nChecking dataset consistency...")

for benchmark in BENCHMARKS:
    indices_per_model = {}
    for model in MODELS:
        key = (model, benchmark)
        if key in all_results:
            indices = all_results[key].get('dataset_indices', [])
            indices_per_model[model] = tuple(indices)
    
    if len(set(indices_per_model.values())) == 1:
        print(f"  ✓ {benchmark}: All models use same {len(indices_per_model[MODELS[0]])} problems")
    else:
        print(f"  ⚠ {benchmark}: Indices differ across models!")

---
## 3. Build Summary DataFrame

In [ ]:
# Extract key metrics into DataFrame
rows = []

for (model, benchmark), data in all_results.items():
    for condition in CONDITIONS:
        cond_data = data.get('conditions', {}).get(condition, {})
        probe_data = cond_data.get('probe', {})
        
        row = {
            'model': model,
            'model_name': MODEL_NAMES.get(model, model),
            'benchmark': benchmark,
            'condition': condition,
            'n_samples': cond_data.get('n_samples', 0),
            'accuracy': cond_data.get('accuracy'),
            'entropy_mean': cond_data.get('entropy_mean', 0),
            'entropy_std': cond_data.get('entropy_std', 0),
            'dim_eff_global': cond_data.get('dim_eff_global', 0),
            'generated_tokens_mean': cond_data.get('generated_tokens_mean', 0),
            'probe_auroc': probe_data.get('auroc', 0.5) if probe_data else None,
            'probe_auroc_ci_low': probe_data.get('auroc_ci_low', 0) if probe_data else None,
            'probe_auroc_ci_high': probe_data.get('auroc_ci_high', 1) if probe_data else None,
            'probe_balanced_acc': probe_data.get('balanced_acc', 0.5) if probe_data else None,
            'pos_rate': probe_data.get('pos_rate', 0) if probe_data else None,
        }
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Built DataFrame with {len(df)} rows")
df.head(10)

---
## 4. Generate Paper Table

In [ ]:
def generate_latex_table(df: pd.DataFrame) -> str:
    """Generate LaTeX table for paper."""
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Main results across models and benchmarks. Accuracy, pre-collapse entropy $H_{\text{int}}(I)$, effective dimensionality $d_{\text{eff}}$, and probe AUROC with 95\% CI.}")
    lines.append(r"\label{tab:main_results}")
    lines.append(r"\small")
    lines.append(r"\begin{tabular}{llcccc}")
    lines.append(r"\toprule")
    lines.append(r"Model & Benchmark & Acc (B→CoT) & $H_{\text{int}}$ (B→CoT) & $d_{\text{eff}}$ & AUROC [95\% CI] \\")
    lines.append(r"\midrule")
    
    for model in MODELS:
        model_df = df[df['model'] == model]
        model_name = MODEL_NAMES.get(model, model)
        
        for i, benchmark in enumerate(BENCHMARKS):
            bench_df = model_df[model_df['benchmark'] == benchmark]
            
            baseline = bench_df[bench_df['condition'] == 'baseline'].iloc[0] if len(bench_df[bench_df['condition'] == 'baseline']) > 0 else None
            cot = bench_df[bench_df['condition'] == 'enhanced'].iloc[0] if len(bench_df[bench_df['condition'] == 'enhanced']) > 0 else None
            
            if baseline is None or cot is None:
                continue
            
            # Format values
            acc_b = f"{baseline['accuracy']*100:.1f}" if baseline['accuracy'] else '--'
            acc_c = f"{cot['accuracy']*100:.1f}" if cot['accuracy'] else '--'
            ent_b = f"{baseline['entropy_mean']:.2f}"
            ent_c = f"{cot['entropy_mean']:.2f}"
            dim_eff = f"{cot['dim_eff_global']}"
            
            auroc = cot['probe_auroc']
            ci_lo = cot['probe_auroc_ci_low']
            ci_hi = cot['probe_auroc_ci_high']
            auroc_str = f"{auroc:.2f} [{ci_lo:.2f}, {ci_hi:.2f}]" if auroc else '--'
            
            # First row of model gets model name
            if i == 0:
                model_col = model_name
            else:
                model_col = ""
            
            line = f"{model_col} & {benchmark.upper()} & {acc_b}→{acc_c} & {ent_b}→{ent_c} & {dim_eff} & {auroc_str} \\\\"
            lines.append(line)
        
        if model != MODELS[-1]:
            lines.append(r"\midrule")
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

# Generate and save
latex_table = generate_latex_table(df)
print(latex_table)

with open(os.path.join(OUTPUT_PATH, 'table_main_results.tex'), 'w') as f:
    f.write(latex_table)
print(f"\n✓ Saved to {OUTPUT_PATH}/table_main_results.tex")

---
## 5. Generate Figures

In [ ]:
# Figure 1: Accuracy comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax_idx, benchmark in enumerate(BENCHMARKS):
    ax = axes[ax_idx]
    bench_df = df[(df['benchmark'] == benchmark) & (df['condition'].isin(['baseline', 'enhanced']))]
    
    x = np.arange(len(MODELS))
    width = 0.35
    
    baseline_accs = []
    cot_accs = []
    
    for model in MODELS:
        model_df = bench_df[bench_df['model'] == model]
        b = model_df[model_df['condition'] == 'baseline']['accuracy'].values
        c = model_df[model_df['condition'] == 'enhanced']['accuracy'].values
        baseline_accs.append(b[0] if len(b) > 0 and b[0] else 0)
        cot_accs.append(c[0] if len(c) > 0 and c[0] else 0)
    
    bars1 = ax.bar(x - width/2, baseline_accs, width, label='Baseline', color='steelblue')
    bars2 = ax.bar(x + width/2, cot_accs, width, label='CoT', color='coral')
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{benchmark.upper()}')
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_NAMES[m] for m in MODELS], rotation=15, ha='right')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'fig_accuracy.pdf'))
print(f"✓ Saved fig_accuracy.pdf")
plt.show()

In [ ]:
# Figure 2: Entropy comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax_idx, benchmark in enumerate(BENCHMARKS):
    ax = axes[ax_idx]
    bench_df = df[df['benchmark'] == benchmark]
    
    x = np.arange(len(MODELS))
    width = 0.25
    
    for i, condition in enumerate(CONDITIONS):
        cond_df = bench_df[bench_df['condition'] == condition]
        entropies = [cond_df[cond_df['model'] == m]['entropy_mean'].values[0] if len(cond_df[cond_df['model'] == m]) > 0 else 0 for m in MODELS]
        colors = {'baseline': 'steelblue', 'enhanced': 'coral', 'babble': 'gray'}
        ax.bar(x + (i - 1) * width, entropies, width, label=condition.capitalize(), color=colors[condition])
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Entropy (bits)')
    ax.set_title(f'{benchmark.upper()}')
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_NAMES[m] for m in MODELS], rotation=15, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'fig_entropy.pdf'))
print(f"✓ Saved fig_entropy.pdf")
plt.show()

In [ ]:
# Figure 3: Probe AUROC with error bars
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax_idx, benchmark in enumerate(BENCHMARKS):
    ax = axes[ax_idx]
    bench_df = df[(df['benchmark'] == benchmark) & (df['condition'] == 'enhanced')]
    
    aurocs = []
    ci_lows = []
    ci_highs = []
    
    for model in MODELS:
        model_row = bench_df[bench_df['model'] == model]
        if len(model_row) > 0:
            aurocs.append(model_row['probe_auroc'].values[0] or 0.5)
            ci_lows.append(model_row['probe_auroc_ci_low'].values[0] or 0)
            ci_highs.append(model_row['probe_auroc_ci_high'].values[0] or 1)
        else:
            aurocs.append(0.5)
            ci_lows.append(0)
            ci_highs.append(1)
    
    x = np.arange(len(MODELS))
    yerr = [[a - l for a, l in zip(aurocs, ci_lows)], [h - a for a, h in zip(aurocs, ci_highs)]]
    
    bars = ax.bar(x, aurocs, yerr=yerr, capsize=5, color='coral', edgecolor='black')
    ax.axhline(0.5, color='gray', linestyle='--', label='Chance')
    
    ax.set_xlabel('Model')
    ax.set_ylabel('AUROC')
    ax.set_title(f'{benchmark.upper()} - CoT Probe')
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_NAMES[m] for m in MODELS], rotation=15, ha='right')
    ax.set_ylim(0.4, 1.0)
    ax.legend(loc='lower right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'fig_probe_auroc.pdf'))
print(f"✓ Saved fig_probe_auroc.pdf")
plt.show()

In [ ]:
# Figure 4: CoT improvement delta
fig, ax = plt.subplots(figsize=(10, 5))

deltas = []
labels = []

for model in MODELS:
    for benchmark in BENCHMARKS:
        model_bench_df = df[(df['model'] == model) & (df['benchmark'] == benchmark)]
        baseline = model_bench_df[model_bench_df['condition'] == 'baseline']['accuracy'].values
        cot = model_bench_df[model_bench_df['condition'] == 'enhanced']['accuracy'].values
        
        if len(baseline) > 0 and len(cot) > 0 and baseline[0] and cot[0]:
            delta = (cot[0] - baseline[0]) * 100
            deltas.append(delta)
            labels.append(f"{MODEL_NAMES[model]}\n{benchmark.upper()}")

colors = ['coral' if d > 0 else 'steelblue' for d in deltas]
ax.barh(range(len(deltas)), deltas, color=colors)
ax.set_yticks(range(len(deltas)))
ax.set_yticklabels(labels)
ax.set_xlabel('Accuracy Improvement (CoT - Baseline) %')
ax.set_title('Effect of Chain-of-Thought')
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'fig_cot_improvement.pdf'))
print(f"✓ Saved fig_cot_improvement.pdf")
plt.show()

---
## 6. Export Data

In [ ]:
# Save full DataFrame as CSV
df.to_csv(os.path.join(OUTPUT_PATH, 'summary_all_experiments.csv'), index=False)
print(f"✓ Saved summary_all_experiments.csv")

# Save CoT improvements
improvements = []
for model in MODELS:
    for benchmark in BENCHMARKS:
        model_bench_df = df[(df['model'] == model) & (df['benchmark'] == benchmark)]
        baseline = model_bench_df[model_bench_df['condition'] == 'baseline']
        cot = model_bench_df[model_bench_df['condition'] == 'enhanced']
        
        if len(baseline) > 0 and len(cot) > 0:
            b_acc = baseline['accuracy'].values[0]
            c_acc = cot['accuracy'].values[0]
            b_ent = baseline['entropy_mean'].values[0]
            c_ent = cot['entropy_mean'].values[0]
            
            improvements.append({
                'model': model,
                'benchmark': benchmark,
                'baseline_acc': b_acc,
                'cot_acc': c_acc,
                'acc_delta': (c_acc - b_acc) if b_acc and c_acc else None,
                'baseline_entropy': b_ent,
                'cot_entropy': c_ent,
                'entropy_delta': c_ent - b_ent,
                'cot_auroc': cot['probe_auroc'].values[0],
            })

improvements_df = pd.DataFrame(improvements)
improvements_df.to_csv(os.path.join(OUTPUT_PATH, 'cot_improvements.csv'), index=False)
print(f"✓ Saved cot_improvements.csv")

improvements_df

---
## 7. Summary Statistics

In [ ]:
print("="*60)
print("SUMMARY STATISTICS")
print("="*60)

# Overall accuracy improvement
cot_df = df[df['condition'] == 'enhanced']
baseline_df = df[df['condition'] == 'baseline']

avg_baseline_acc = baseline_df['accuracy'].dropna().mean()
avg_cot_acc = cot_df['accuracy'].dropna().mean()
print(f"\nAverage Accuracy:")
print(f"  Baseline: {avg_baseline_acc:.1%}")
print(f"  CoT: {avg_cot_acc:.1%}")
print(f"  Improvement: +{(avg_cot_acc - avg_baseline_acc)*100:.1f}pp")

# Entropy reduction
avg_baseline_ent = baseline_df['entropy_mean'].mean()
avg_cot_ent = cot_df['entropy_mean'].mean()
print(f"\nAverage Entropy:")
print(f"  Baseline: {avg_baseline_ent:.2f} bits")
print(f"  CoT: {avg_cot_ent:.2f} bits")
print(f"  Reduction: {(avg_baseline_ent - avg_cot_ent):.2f} bits")

# Probe performance
avg_auroc = cot_df['probe_auroc'].dropna().mean()
print(f"\nProbe AUROC (CoT):")
print(f"  Average: {avg_auroc:.3f}")
print(f"  Range: [{cot_df['probe_auroc'].dropna().min():.3f}, {cot_df['probe_auroc'].dropna().max():.3f}]")

---
## Done! 🎉

Paper outputs saved to: `Google Drive/intention_collapse_v3/paper_outputs/`